# Complete Feature Engineering Workflow

## What is a Complete Feature Engineering Workflow?

A complete feature engineering workflow is a systematic process of converting raw data into meaningful and useful features for machine learning.

It combines different feature engineering and feature selection techniques into one complete process.

The objective is to prepare high-quality features that help machine learning models learn patterns and make accurate predictions.

## Steps in a Complete Feature Engineering Workflow

### Step 1: Understand the Business Problem

Understand what the business wants to predict.

Example:
A telecom company wants to predict whether a customer will leave its services.

Target variable: Churn.

### Step 2: Collect Raw Data

Collect the required information from available datasets.

Example:
- Customer age
- Tenure
- Monthly charges
- Total charges
- Contract type
- Support calls

### Step 3: Inspect the Dataset

Understand the dataset before modifying it.

Check:
- Dataset shape
- Data types
- Missing values
- Duplicate records
- Unique values
- Statistical summaries

### Step 4: Clean the Data

Handle:
- Missing values
- Duplicate records
- Incorrect data types
- Invalid values
- Inconsistent categories

### Step 5: Create New Features

Apply feature engineering techniques.

Examples:
- Numerical feature engineering
- Categorical feature engineering
- Interaction features
- Aggregation features
- Binning and discretization
- Date and time features
- Text features
- Domain-based features

### Step 6: Select Important Features

Remove unnecessary or less useful features.

Methods:
- Correlation analysis
- Variance threshold
- Statistical tests
- Feature importance
- Recursive Feature Elimination

### Step 7: Prevent Feature Leakage

Ensure that the model does not receive information from the future or the target outcome.

Split the dataset before fitting preprocessing and feature selection transformations.

### Step 8: Preprocess the Features

Apply:
- Missing value imputation
- Numerical scaling
- Categorical encoding

### Step 9: Train the Machine Learning Model

Use the prepared features to train a suitable model.

Examples:
- Logistic Regression
- Decision Tree
- Random Forest
- Support Vector Machine

### Step 10: Evaluate the Model

Evaluate the model using unseen test data.

Common metrics:
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

## Why is a Complete Workflow Important?

1. Organizes the entire machine learning process.
2. Improves data quality.
3. Creates meaningful features.
4. Reduces unnecessary information.
5. Prevents data leakage.
6. Improves model reliability.
7. Makes the process reusable.
8. Simplifies deployment.

## Final Rule

Feature engineering should not stop after creating new columns. Every feature must be inspected, justified, validated, and evaluated before being used in a machine learning model.

# Demonstration of Complete Feature Engineering

## Business Scenario: Telecom Customer Churn Prediction

A telecom company wants to identify customers who may stop using its services.

The company has collected customer information such as:

| Feature | Description |
|---|---|
| CustomerID | Unique customer identifier |
| Age | Customer age |
| Tenure | Number of months as a customer |
| MonthlyCharges | Monthly bill amount |
| TotalCharges | Total customer charges |
| SupportCalls | Number of support calls |
| ContractType | Customer contract category |
| InternetService | Internet service category |
| PhoneService | Whether phone service is active |
| Churn | Target variable |

## Features We Will Create

| New Feature | Source Columns | Business Purpose |
|---|---|---|
| Average_Monthly_Spending | TotalCharges, Tenure | Understand average customer spending |
| Customer_Loyalty_Group | Tenure | Identify customer relationship duration |
| High_Value_Customer | MonthlyCharges | Identify higher revenue customers |
| Support_Issue_Level | SupportCalls | Identify possible service problems |
| Service_Count | InternetService, PhoneService | Understand service adoption |
| Charges_Per_Service | MonthlyCharges, Service_Count | Understand average charges per subscribed service |

## Complete Implementation

We will:

1. Create a sample telecom dataset.
2. Inspect and validate the data.
3. Create domain-based features.
4. Separate features and target.
5. Split training and testing data.
6. Apply numerical and categorical preprocessing.
7. Perform feature selection.
8. Train a machine learning model.
9. Evaluate model performance.
10. Predict churn for a new customer.

## Expected Outcome

At the end, we will have:

- A cleaned dataset.
- A dataset containing engineered features.
- Selected ML-ready features.
- A trained machine learning model.
- Model evaluation results.
- A prediction for a new customer.

In [1]:
# ==============================================================
# NOTEBOOK 15: COMPLETE FEATURE ENGINEERING WORKFLOW
# Business Scenario: Telecom Customer Churn Prediction
# ==============================================================


# STEP 1: IMPORT REQUIRED LIBRARIES

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")


# STEP 2: CREATE RAW TELECOM DATASET

np.random.seed(42)

n = 500

data = pd.DataFrame({
    "CustomerID": range(1, n + 1),
    "Age": np.random.randint(18, 70, n),
    "Tenure": np.random.randint(1, 73, n),
    "MonthlyCharges": np.random.uniform(20, 120, n).round(2),
    "SupportCalls": np.random.randint(0, 10, n),
    "ContractType": np.random.choice(
        ["Monthly", "Yearly", "Two Year"], n
    ),
    "InternetService": np.random.choice(
        ["DSL", "Fiber", "No"], n
    ),
    "PhoneService": np.random.choice(
        ["Yes", "No"], n
    )
})

# Create total charges with some variation

data["TotalCharges"] = (
    data["Tenure"] * data["MonthlyCharges"]
    + np.random.normal(0, 50, n)
).clip(lower=0).round(2)

# Generate churn target

churn_probability = (
    0.10
    + (data["Tenure"] < 12) * 0.20
    + (data["MonthlyCharges"] > 90) * 0.15
    + (data["SupportCalls"] > 5) * 0.20
    + (data["ContractType"] == "Monthly") * 0.15
)

churn_probability = churn_probability.clip(0, 0.95)

data["Churn"] = np.random.binomial(
    1, churn_probability
)


# STEP 3: INSPECT THE RAW DATASET

print("\n========== RAW DATA INSPECTION ==========")

print("\nFirst Five Records:")
print(data.head())

print("\nDataset Shape:")
print(data.shape)

print("\nData Types:")
print(data.dtypes)

print("\nMissing Values:")
print(data.isnull().sum())

print("\nDuplicate Records:")
print(data.duplicated().sum())

print("\nStatistical Summary:")
print(data.describe())


# STEP 4: HANDLE DUPLICATE RECORDS

data = data.drop_duplicates().copy()

print("\nDataset Shape After Duplicate Removal:")
print(data.shape)


# STEP 5: CREATE DOMAIN-BASED FEATURES

# Feature 1: Average Monthly Spending

data["Average_Monthly_Spending"] = np.where(
    data["Tenure"] > 0,
    data["TotalCharges"] / data["Tenure"],
    np.nan
)

# Feature 2: Customer Loyalty Group

data["Customer_Loyalty_Group"] = pd.cut(
    data["Tenure"],
    bins=[0, 12, 36, np.inf],
    labels=["New", "Regular", "Loyal"],
    include_lowest=True
)

# Feature 3: High Value Customer

data["High_Value_Customer"] = np.where(
    data["MonthlyCharges"] > 80,
    "Yes",
    "No"
)

# Feature 4: Support Issue Level

data["Support_Issue_Level"] = pd.cut(
    data["SupportCalls"],
    bins=[-1, 2, 5, np.inf],
    labels=["Low", "Medium", "High"]
)

# Feature 5: Service Count

data["Service_Count"] = (
    (data["InternetService"] != "No").astype(int)
    + (data["PhoneService"] == "Yes").astype(int)
)

# Feature 6: Charges Per Service

data["Charges_Per_Service"] = (
    data["MonthlyCharges"] /
    data["Service_Count"].replace(0, np.nan)
)

print("\nEngineered Features Created Successfully!")


# STEP 6: DISPLAY ENGINEERED DATASET

print("\nFirst Five Engineered Records:")
print(data.head())

print("\nEngineered Dataset Shape:")
print(data.shape)


# STEP 7: CHECK ENGINEERED FEATURES

engineered_features = [
    "Average_Monthly_Spending",
    "Customer_Loyalty_Group",
    "High_Value_Customer",
    "Support_Issue_Level",
    "Service_Count",
    "Charges_Per_Service"
]

print("\nEngineered Feature Summary:")

print(data[engineered_features].head())

print("\nMissing Values in Engineered Features:")
print(data[engineered_features].isnull().sum())


# STEP 8: SEPARATE FEATURES AND TARGET

# CustomerID is an identifier and is excluded.
# Churn is the target and must not be used as an input feature.

X = data.drop(
    columns=["CustomerID", "Churn"]
)

y = data["Churn"]

print("\nFeature Dataset Shape:")
print(X.shape)

print("Target Dataset Shape:")
print(y.shape)


# STEP 9: IDENTIFY NUMERICAL AND CATEGORICAL FEATURES

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)


# STEP 10: SPLIT DATA INTO TRAINING AND TESTING SETS

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Dataset Shape:")
print(X_train.shape)

print("Testing Dataset Shape:")
print(X_test.shape)


# STEP 11: CREATE NUMERICAL PREPROCESSING PIPELINE

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# STEP 12: CREATE CATEGORICAL PREPROCESSING PIPELINE

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# STEP 13: COMBINE PREPROCESSING USING COLUMNTRANSFORMER

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])


# STEP 14: CREATE COMPLETE MACHINE LEARNING PIPELINE

complete_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectKBest(
        score_func=f_classif,
        k=8
    )),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

print("\nComplete Machine Learning Pipeline Created!")


# STEP 15: TRAIN THE MODEL

complete_pipeline.fit(
    X_train,
    y_train
)

print("\nModel Trained Successfully!")


# STEP 16: MAKE TEST DATA PREDICTIONS

predictions = complete_pipeline.predict(X_test)

print("\nFirst 10 Predictions:")
print(predictions[:10])

print("\nActual Values:")
print(y_test.iloc[:10].values)


# STEP 17: EVALUATE MODEL PERFORMANCE

accuracy = accuracy_score(
    y_test,
    predictions
)

print("\n========== MODEL EVALUATION ==========")

print("Model Accuracy:", round(accuracy, 4))

print("\nClassification Report:")

print(classification_report(
    y_test,
    predictions,
    zero_division=0
))

print("\nConfusion Matrix:")

print(confusion_matrix(
    y_test,
    predictions
))


# STEP 18: DISPLAY SELECTED FEATURES

preprocessor_fitted = complete_pipeline.named_steps[
    "preprocessing"
]

feature_names = preprocessor_fitted.get_feature_names_out()

selected_indices = complete_pipeline.named_steps[
    "feature_selection"
].get_support(indices=True)

selected_features = feature_names[selected_indices]

print("\nSelected Features:")

for feature in selected_features:
    print("-", feature)


# STEP 19: CREATE NEW CUSTOMER DATA

new_customer = pd.DataFrame({
    "Age": [30],
    "Tenure": [8],
    "MonthlyCharges": [95.50],
    "SupportCalls": [6],
    "ContractType": ["Monthly"],
    "InternetService": ["Fiber"],
    "PhoneService": ["Yes"],
    "TotalCharges": [764.00],
    "Average_Monthly_Spending": [95.50],
    "Customer_Loyalty_Group": ["New"],
    "High_Value_Customer": ["Yes"],
    "Support_Issue_Level": ["High"],
    "Service_Count": [2],
    "Charges_Per_Service": [47.75]
})

# Ensure the new customer has the same feature order as training data

new_customer = new_customer[X.columns]


# STEP 20: PREDICT NEW CUSTOMER CHURN

new_prediction = complete_pipeline.predict(new_customer)

print("\n========== NEW CUSTOMER PREDICTION ==========")

if new_prediction[0] == 1:
    print("Prediction: Customer is likely to Churn.")
else:
    print("Prediction: Customer is likely to Stay.")


# STEP 21: FINAL WORKFLOW SUMMARY

print("\n========== FINAL WORKFLOW SUMMARY ==========")

print("Original Dataset Records:", len(data))

print("Engineered Features Created:", len(engineered_features))

print("Training Records:", len(X_train))

print("Testing Records:", len(X_test))

print("Selected ML Features:", len(selected_features))

print("Model Accuracy:", round(accuracy, 4))

print("\nComplete Feature Engineering Workflow Finished Successfully!")

Libraries imported successfully!

========== RAW DATA INSPECTION ==========

First Five Records:
   CustomerID  Age  Tenure  MonthlyCharges  SupportCalls ContractType  \
0           1   56      16           76.99             1      Monthly   
1           2   69      61           96.25             1       Yearly   
2           3   46      64          107.68             5     Two Year   
3           4   32      63           54.21             8      Monthly   
4           5   60      69          102.13             0      Monthly   

  InternetService PhoneService  TotalCharges  Churn  
0             DSL           No       1242.80      0  
1           Fiber           No       5935.71      1  
2           Fiber          Yes       6912.79      0  
3             DSL          Yes       3526.17      0  
4              No           No       6968.72      0  

Dataset Shape:
(500, 10)

Data Types:
CustomerID           int64
Age                  int32
Tenure               int32
MonthlyCharges     f

C:\Users\Akalya Senthil\AppData\Local\Temp\ipykernel_7232\2857952920.py:210: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(



Model Trained Successfully!

First 10 Predictions:
[0 1 0 0 0 0 1 0 0 0]

Actual Values:
[0 0 0 0 1 0 1 0 0 0]

========== MODEL EVALUATION ==========
Model Accuracy: 0.62

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.76      0.73        67
           1       0.41      0.33      0.37        33

    accuracy                           0.62       100
   macro avg       0.55      0.55      0.55       100
weighted avg       0.60      0.62      0.61       100


Confusion Matrix:
[[51 16]
 [22 11]]

Selected Features:
- numerical__MonthlyCharges
- numerical__Average_Monthly_Spending
- categorical__Customer_Loyalty_Group_Loyal
- categorical__Customer_Loyalty_Group_New
- categorical__High_Value_Customer_No
- categorical__High_Value_Customer_Yes
- categorical__Support_Issue_Level_High
- categorical__Support_Issue_Level_Low

========== NEW CUSTOMER PREDICTION ==========
Prediction: Customer is likely to Churn.

========== FINAL WORK

# 4. Feature Justification

## Feature Engineering Documentation

| Feature Name | Source Columns | Feature Logic / Formula | Reason for Creation | Expected Relationship with Churn | Business Meaning | Potential ML Value | Leakage Risk | Final Decision |
|---|---|---|---|---|---|---|---|---|
| Average_Monthly_Spending | TotalCharges, Tenure | TotalCharges / Tenure | Understand average customer spending | May vary with customer spending behaviour | Average monthly spending | Medium | Low if available at prediction time | Retain |
| Customer_Loyalty_Group | Tenure | Group tenure into New, Regular, Loyal | Understand customer relationship duration | New customers may have higher churn | Customer loyalty segment | High | Low | Retain |
| High_Value_Customer | MonthlyCharges | MonthlyCharges > 80 | Identify higher revenue customers | Higher charges may increase churn | High monthly revenue segment | Medium | Low | Retain |
| Support_Issue_Level | SupportCalls | Group support calls into Low, Medium, High | Understand service issues | High support activity may increase churn | Customer service experience | High | Low | Retain |
| Service_Count | InternetService, PhoneService | Count active services | Understand service adoption | Relationship may vary | Number of subscribed services | Medium | Low | Retain |
| Charges_Per_Service | MonthlyCharges, Service_Count | MonthlyCharges / Service_Count | Understand average charges per service | Relationship may vary | Average charges per subscribed service | Medium | Low | Retain |

## Feature Selection Justification

The workflow uses `SelectKBest` with the ANOVA F-test to select eight features.

This helps identify features with stronger statistical relationships with the churn target.

However, feature selection results must be validated using model performance and business knowledge.

## Leakage Prevention

The workflow follows these practices:

1. Separates input features and target.
2. Splits training and testing data before fitting transformations.
3. Fits missing-value imputation using training data.
4. Fits numerical scaling using training data.
5. Fits categorical encoding using training data.
6. Performs feature selection using training data.
7. Evaluates the model using unseen test data.

## Before and After Feature Engineering

### Before Feature Engineering

The dataset contains basic customer information.

Examples:
- Tenure
- MonthlyCharges
- SupportCalls
- ContractType

### After Feature Engineering

The dataset contains additional business-related information.

Examples:
- Average_Monthly_Spending
- Customer_Loyalty_Group
- High_Value_Customer
- Support_Issue_Level
- Service_Count
- Charges_Per_Service

These features provide additional ways to understand customer behaviour.

## Final Learning

A complete feature engineering workflow connects data preparation, business knowledge, feature creation, feature selection, and machine learning.

The goal is not to create the maximum number of features. The goal is to create meaningful, reliable, and useful features.

**Final Rule:** Every engineered feature should have a clear purpose, a valid calculation, business justification, and measurable value for machine learning.